**Why does this matter for quant finance**

Every single risk report, performance summary, and portfolio analysis you'll ever build boils down to aggregating numbers meaningfully. Volatility is just standard deviation of returns. VaR (Value at Risk) is a percentile. Cumulative performance is a running sum. NumPy gives you all of these in one line — and understanding the axis argument is what separates someone who uses these functions from someone who understands them.

**<h3>Part 1 - The Axis Argument<h3>**

In [1]:
import numpy as np

# 5 days × 3 stocks returns matrix
returns = np.array([[ 0.01,  0.02, -0.01],
                    [-0.02,  0.01,  0.03],
                    [ 0.03, -0.01,  0.02],
                    [ 0.00,  0.03, -0.02],
                    [ 0.02,  0.01,  0.01]])
# shape: (5, 3)

# No axis → single value across EVERYTHING
print(np.mean(returns))         # mean of all 15 values

# axis=0 → collapse DOWN rows → one value per COLUMN (per stock)
print(np.mean(returns, axis=0)) # shape (3,) — mean return per stock

# axis=1 → collapse ACROSS columns → one value per ROW (per day)
print(np.mean(returns, axis=1)) # shape (5,) — mean return per day

0.008666666666666666
[0.008 0.012 0.006]
[0.00666667 0.00666667 0.01333333 0.00333333 0.01333333]


Stock0  Stock1  Stock2
Day0  [  0.01    0.02   -0.01  ]
Day1  [ -0.02    0.01    0.03  ]
Day2  [  0.03   -0.01    0.02  ]  axis=0 ↓ (per stock)
Day3  [  0.00    0.03   -0.02  ]
Day4  [  0.02    0.01    0.01  ]
         ────    ────    ────
        mean0   mean1   mean2

axis=1 → (per day) each row collapses to one value

**<h3>Part 2 - The Core Aggregation Functions<h3>**

Using:

np.sum() and np.mean()<br>
np.std() and np.var() -- for Volatility<br>
np.min(), np.max(), np.median()<br>
np.percentile() -- for VaR<br>
np.cumsum() -- for Cumulative returns<br>
np.argmin() & np.argmax() —- Finding When Events Happened<br>

**np.sum() and np.mean()**

In [ ]:
# Total return per stock across all days
print(np.sum(returns, axis=0))   # [ 0.04  0.06  0.03]

# Average daily return per stock
print(np.mean(returns, axis=0))  # [ 0.008  0.012  0.006]

[0.04 0.06 0.03]
[0.008 0.012 0.006]


**np.std() & np.var() — Volatility**

In [93]:
# Standard deviation per stock — this IS volatility in quant finance
volatility = np.std(returns, axis=0)
print("Volatility per stock:", volatility)

# Variance = std² — used in portfolio theory (covariance matrices etc.)
variance = np.var(returns, axis=0)
print("Variance per stock:", variance)

Volatility per stock: [0.00894592 0.01001363 0.01048125]
Variance per stock: [8.00295123e-05 1.00272701e-04 1.09856597e-04]


‼️ 🏦 Quant context: Annualised volatility = np.std(daily_returns) * np.sqrt(252) — one of the most used formulas in finance. The 252 is the number of trading days in a year.

In [5]:
annualised_vol = np.std(returns, axis=0) * np.sqrt(252)
print("Annualised volatility:", annualised_vol)

Annualised volatility: [0.27311536 0.21059915 0.29442826]


**np.min(), np.max(), np.median()**

In [6]:
print(np.min(returns, axis=0))     # worst day per stock
print(np.max(returns, axis=0))     # best day per stock
print(np.median(returns, axis=0))  # median daily return per stock

[-0.02 -0.01 -0.02]
[0.03 0.03 0.03]
[0.01 0.01 0.01]


**np.percentile() — The VaR Function**

In [7]:
# 5th percentile return per stock — this is essentially 95% daily VaR!
var_95 = np.percentile(returns, 5, axis=0)
print("95% VaR per stock:", var_95)

95% VaR per stock: [-0.016 -0.006 -0.018]


    🏦 Quant context: VaR (Value at Risk) answers "what's the worst loss I'd expect on 95% of trading days?" — directly computed with np.percentile(returns, 5).

**np.cumsum() — Cumulative Returns**

In [8]:
# Cumulative return over time per stock
cumulative = np.cumsum(returns, axis=0)
print(cumulative)
# Each row shows the running total up to that day

[[ 0.01  0.02 -0.01]
 [-0.01  0.03  0.02]
 [ 0.02  0.02  0.04]
 [ 0.02  0.05  0.02]
 [ 0.04  0.06  0.03]]


Visualised for Stock 0: [0.01, -0.01, 0.02, 0.02, 0.04] — a running P&L curve. This is the basis of equity curves in backtesting. 📈

**np.argmin() & np.argmax() — Finding When Events Happened**

In [10]:
# Which DAY had the best and worst return per stock?
best_day = np.argmax(returns, axis=0)
worst_day = np.argmin(returns, axis=0)
print("Best day index per stock:", best_day)
print("Worst day index per stock:", worst_day)

Best day index per stock: [2 3 1]
Worst day index per stock: [1 2 3]


🏦 Quant context: argmax on a cumulative returns curve gives you the peak before a drawdown — critical for drawdown analysis.

⚡ Part 3 — np.sort() & np.argsort()

In [95]:
daily_returns = np.array([0.02, -0.03, 0.01, -0.01, 0.04, -0.02])

# Sort returns (ascending — worst to best)
sorted_returns = np.sort(daily_returns)
print(sorted_returns)  # [-0.03 -0.02 -0.01  0.01  0.02  0.04]

# argsort — returns the INDICES that would sort the array
order = np.argsort(daily_returns)
print(order)  # [1 5 3 2 0 4] — index 1 was worst, index 4 was best

print("")
print("Sum: ", np.sum(daily_returns, axis=0))
print("Cum Sum: ", np.cumsum(daily_returns, axis=0))

[-0.03 -0.02 -0.01  0.01  0.02  0.04]
[1 5 3 2 0 4]

Sum:  0.010000000000000002
Cum Sum:  [ 2.00000000e-02 -1.00000000e-02  1.73472348e-18 -1.00000000e-02
  3.00000000e-02  1.00000000e-02]


**<h3>Tasks<h3>**

<h6># Task 1: Calculate mean and annualised volatility per stock<br>
# Task 2: Calculate the 95% VaR per stock (5th percentile)<br>
# Task 3: Build the cumulative return series per stock using np.cumsum()<br>
# Task 4: Find which day had the maximum cumulative return per stock (argmax)<br>
# Task 5: Build a 50/25/25 weighted portfolio and calculate its daily returns<br>
#                   weights = np.array([0.5, 0.25, 0.25])<h6>

In [70]:
# Simulate 1 year (252 days) of daily returns for 3 stocks
np.random.seed(42)  # set seed for reproducibility — important in quant work!
tasks_returns = np.random.randn(252, 3) * 0.01

In [79]:
#print(tasks_returns)
print(np.sum(tasks_returns, axis=1))

[ 1.00613839e-02  1.05473952e-02  1.87717316e-02 -3.86587403e-03
 -3.39623581e-02 -1.26087132e-02 -8.54679008e-03 -1.58299628e-02
 -1.58445371e-02 -5.16634421e-03  1.23707435e-02 -1.45600967e-02
 -3.07899258e-02  1.10669610e-02 -1.89527397e-02 -1.23360753e-03
 -1.09533790e-02 -4.50327992e-03  1.12306212e-02  9.97596183e-03
 -1.77116819e-02  9.72559227e-03  1.29315880e-02  1.25431242e-02
 -1.09092749e-02  6.09942222e-03 -2.11548003e-02  1.31673640e-02
 -3.94848529e-03  3.12258339e-03  3.63669446e-03 -2.18328525e-02
  5.62289006e-03 -2.07060320e-02 -1.30627750e-02  2.46481457e-02
 -1.73566674e-02  2.49695845e-02  7.44746078e-04  7.26077810e-03
  1.28443880e-02  1.37546166e-02 -1.45718269e-02 -1.98557611e-02
 -1.50813552e-02  4.44619596e-03 -1.89887164e-03 -1.15706622e-03
 -1.95245045e-03 -5.01530374e-03 -8.30836617e-04 -1.89025248e-03
  1.14830393e-02  4.68956542e-03  1.30128939e-02  3.13163407e-02
 -1.88863871e-02 -5.51760020e-03  1.11687594e-02  3.90904641e-02
 -1.30238271e-02  9.73010

In [ ]:
#Task 1 - Calculating mean and annualised volatility per stock
print("Task 1")
print(np.shape(tasks_returns)) #252 rows days, 3 columns stocks
#per stock - thus axis should be by column - i.e 0

print("Mean per stock: ", np.mean(tasks_returns, axis=0)) #mean
#print("Annualised Volatility per stock: ",np.std(tasks_returns, axis=0)) #volatility using the std dev #WRONG to make it annualised you have to multply it by the sqrt of 252
print("Annualised Volatility per stock: ",np.std(tasks_returns, axis=0)*np.sqrt(252)) # CORRECT VERSION. This is because volatility is the sqrt of variance 

#Task 2 - Calculating the 95% VaR per stock (5th Percentile)
print("\nTask 2")
print("Returns at the 5th Percentile, i.e what is the worst loss you can expect on 95% of trading days: ", np.percentile(tasks_returns, 5, axis=0))

#Task 3 - Building the cumulative returns series per stock using np.cumsum()
print("\nTask 3")
cumulative_tasks_returns = np.cumsum(tasks_returns, axis=0)
print("Cumulative Returns for each stock: \n", cumulative_tasks_returns) #------ UNPRINT THIS WHEN YOU NEED TO
change_between_cum_tasks_returns = np.diff(cumulative_tasks_returns, axis=0) # THIS ACTUALLY JUST RECOVERS THE ORIGINAL DAILY RETURN
# thus, avg_rate is = np.mean(tasks_returns, axis=0)
avg_rate = np.mean(change_between_cum_tasks_returns, axis=0)
#Average rate of change
print("\nAverage Rate of Change: ", avg_rate)


#Task 4 - Find which day had the maximum cumulative return, per stock
print("\nTask 4")
best_day_tasks_returns = np.argmax(cumulative_tasks_returns, axis=0)
print("Best Day for cumulative returns for each stock: ", best_day_tasks_returns)
stock=0
for day in best_day_tasks_returns:
    #print(stock)
    print(f"Stock {stock+1}'s best day for cumulative returns was Day: {best_day_tasks_returns[stock]}")
    stock+=1

# The below is the SAME as above. (row, column) 
for stock, day in enumerate(best_day_tasks_returns):
    print(f"Stock {stock+1}'s best day was Day: {day}")


#Task 5 -Build a 50/25/25 weighted portfolio and calculate it's daily return
print("\nTask 5")
weights = np.array([0.5,0.25,0.25])
weighted_tasks_returns = tasks_returns * weights
print("This is the shape of the array of weighted returns for each stock, across 252 days: ", np.shape(weighted_tasks_returns))
combined_weighted_return = np.sum(weighted_tasks_returns, axis=1)
print("This is the shape of the array of returns for a portfolio of combined, weighted stocks, across 252 days: ", np.shape(combined_weighted_return))
print(combined_weighted_return)

Task 1
(252, 3)
Mean per stock:  [ 0.00059405 -0.00148555  0.00033874]
Annualised Volatility per stock:  [0.1420121  0.15896138 0.16638468]

Task 2
Returns at the 5th Percentile, i.e what is the worst loss you can expect on 95% of trading days:  [-0.01346202 -0.01646381 -0.01628231]

Task 3
Cumulative Returns for each stock: 
 [[ 4.96714153e-03 -1.38264301e-03  6.47688538e-03]
 [ 2.01974401e-02 -3.72417676e-03  4.13551581e-03]
 [ 3.59895682e-02  3.95017053e-03 -5.59228048e-04]
 [ 4.14151687e-02 -6.84006396e-04 -5.21652558e-03]
 [ 4.38347914e-02 -1.98168088e-02 -2.24657039e-02]
 [ 3.82119161e-02 -2.99451200e-02 -1.93232306e-02]
 [ 2.91316754e-02 -4.40681571e-02 -4.66674289e-03]
 [ 2.68739123e-02 -4.33928750e-02 -1.89142248e-02]
 [ 2.14300851e-02 -4.22836491e-02 -3.04241605e-02]
 [ 2.51870653e-02 -4.82900360e-02 -3.33410980e-02]
 [ 1.91699992e-02 -2.97672542e-02 -3.34760703e-02]
 [ 8.59288987e-03 -2.15418050e-02 -4.56845068e-02]
 [ 1.06815258e-02 -4.11385063e-02 -5.89663673e-02]
 [ 1.265

**Notes**

#Task 5 -Build a 50/25/25 weighted portfolio and calculate it's daily return

1. I have 3 stocks, at 3 different weights. Each stock has it's own return profile
2. Weight each stock
2. add each column together across all days

**🧪 Self-Check Questions**

Q1: What's the difference between np.std() and np.var()? When would you use each?<br>
np.std() gets you the standard deviation, i.e the sqr root of variance from the mean, it is sqr rooted to reflect the variance in the original units of measure, making it easier to interpret. It is used to describe how far data points are from the mean and identifying outliers On the otherhand, np.var() gets you the variance which is the average of sqrd differences from the mean, and because it squares them it maginifies the differences

<br>Q2: What does axis=0 vs axis=1 give you when calling np.sum() on a (252, 3) returns matrix?<br>
axis=0 calculates vertically (down the columns), whereas axis=1 calculates horizontally (across the rows). Thus, np.sum(returns, axis=0) will give you the sum of returns for each stock across the 252 days, where as np.sum(returns, axis=1) will give you the sum of returns for all stocks for each day. 

<br>Q3: Why do we multiply daily volatility by np.sqrt(252) to annualise it?<br>
We multipy daily volatility by the square root of 252 to annualise it because volatility is measured by variance.

<br>Q4: What's the financial interpretation of np.percentile(returns, 5)?<br>
This would give us the return in the bottom 5% of all returns. I.e the worst possible return in 95% of trading days.

<br>Q5: What does np.cumsum() represent in a financial context?<br>
np.cumsum() represents the cumulative sum of say, returns. 

**Claude Review of my answers to self-check question**

Really solid effort Clark! Let me go through each one: 🔍

**Q1 — ✅ Strong answer!** You nailed the key distinction — std is in the original units making it interpretable, variance magnifies differences by squaring. One small addition: variance is more commonly used in **portfolio theory** (covariance matrices, portfolio variance calculations) precisely *because* it's additive across assets — std isn't. So both have their place!

**Q2 — ✅ Perfect.** Couldn't have said it better myself.

**Q3 — ⚠️ You started well but stopped short!**
You said *"because volatility is measured by variance"* — but didn't finish the reasoning. The full answer:

> Variance scales **linearly** with time. So annual variance = daily variance × 252. But since volatility is the **square root** of variance, we take `√252` rather than just `252`. 

The square root is the crucial step that's missing from your answer!

**Q4 — ✅ Correct!** Solid financial interpretation — that's literally how historical VaR is defined.

**Q5 — ✅ Correct but could be richer!** A stronger answer would add: it represents the **equity curve** — the running P&L of a strategy over time, which is fundamental to backtesting and drawdown analysis. 📈

---

**Score: 4/5** — Q3 was the only real gap. Send over your task answers! 🚀

**Looking at enumerate from Task 4**
```
for index, value in enumerate(my_array):
    # index = position (0, 1, 2...)
    # value = the actual element

**Answering Module 7 Quiz**

Let's go! 🎯 Think before you run anything!

---

## 🧪 Module 7 Quiz — Stats & Aggregation Functions

---

**Q1.** What shape does this return, and what does it represent financially?

```python
np.std(returns, axis=1)  # returns.shape = (252, 4)
```

---

**Q2.** What's wrong with this code?

```python
annualised_vol = np.std(returns, axis=0) * 252
```

---

**Q3.** What does this represent financially?

```python
np.argmax(np.cumsum(returns, axis=0), axis=0)
```

---

**Q4.** 🏦 **Quant Finance scenario** — You have a `(252, 4)` returns matrix. Write a single line that tells you the **worst single day** across the entire portfolio (all stocks, all days — one number).

---

**Q5.** What's the difference in output between these two?

```python
# Option A
np.sum(returns, axis=0)

# Option B
np.cumsum(returns, axis=0)
```

---

Take your time! 🧠

**My answers and marking**

Q1: This aggregation does the standard deviation but horizontally. if the original shape is (252, 4), it will return a shape of (252,). This could represent the std deviation of a whole portfolio of 4 stocks across 252 trading days

Q2: If you want to get annualised volatility, you will need to squre root the 252. THis is because volatility is measured by variance which is the average squared mean distance.

Q3: This function represents the coordinates of the peak cumulative return, i.e the day. If you wanted the value on this day you can also use np.max() 

Q4: np.argmin(np.cumsum(returns, axis-0), axis=0)

Q5: Option A returns an array of the sum of numbers vertically, with a single number for all stocks, i.e a shape of (4,0). Option B on the otherhand, will return an array of the running total for all stocks across the 252 days, i.e a shape of (252, 4).


********************
**Marking**

Great round Clark! Let me mark these up: 🔍

**Q1 — ✅ Correct on shape, but refine the interpretation!**
`(252,)` is right. But `axis=1` collapses *across stocks*, so each value represents the **dispersion between stocks on a given day** — not portfolio volatility. Portfolio volatility would still use `axis=0`. Subtle but important distinction!

**Q2 — ✅ Correct.** Nailed the reasoning — variance scales linearly, so volatility needs `np.sqrt(252)`.

**Q3 — ✅ Perfect.** The peak cumulative return day per stock — exactly right. Good instinct flagging `np.max()` for the value too! 💡

**Q4 — ⚠️ Two issues!**
```python
np.argmin(np.cumsum(returns, axis=0), axis=0)  # ❌
```
- Typo: `axis-0` instead of `axis=0` 🐍
- `argmin` on `cumsum` finds worst *cumulative* return day — the question asked for worst **single day** across everything. The answer is simply:
```python
np.min(returns)  # ✅ no axis = one number across everything
```

**Q5 — ✅ Correct!** Small typo — `(4,0)` should be `(4,)` — but the reasoning is spot on.

---

## 📊 Final Score: 4/5

Solid performance! The axis=1 interpretation and the no-axis aggregation are worth remembering. Ready for Module 8? 🚀